In [153]:
from typing import Callable

import pandas as pd
import numpy as np

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from torchmetrics import Accuracy

In [150]:
class MNISTDatasetCsv(Dataset):
  def __init__(self, path: str):
    #df = pd.read_csv('../data/train_data.csv', header=None, dtype=float)
    #self.data = torch.tensor(df.values[:, 1:]).float().reshape(df.shape[0], 1, 28, 28)
    #self.labels = torch.tensor(df.iloc[:, 0]).long()
    data = np.load(path)
    self.data = torch.tensor(data[:, 1:]).float().reshape(data.shape[0], 1, 28, 28)
    self.labels = torch.tensor(data[:, 0]).long()

  def __len__(self) -> int:
    return self.labels.shape[0]

  def __getitem__(self, index) -> tuple[torch.Tensor, int]:
    return self.data[index], self.labels[index]


class MNISTClassifier(nn.Module):
  def __init__(self):
    super().__init__()
    self.model = nn.Sequential(
      nn.Conv2d(1, 8, kernel_size=3),
      nn.ReLU(),
      nn.Conv2d(8, 16, kernel_size=3),
      nn.ReLU(),
      nn.Flatten(),
      nn.Linear(9216, 10),  # 10 classes in total.
    )

  def forward(self, x: torch.Tensor):
    return self.model(x)
    

def train(dataloader, model, loss_fn, metrics_fn, optimizer, epoch, device):
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        pred = model(X)
        loss = loss_fn(pred, y)
        accuracy = metrics_fn(pred, y)

        # Backpropagation.
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 10 == 0:
            loss, current = loss.item(), batch
            step = batch // 10 * (epoch + 1)
            #mlflow.log_metric("loss", f"{loss:2f}", step=step)
            #mlflow.log_metric("accuracy", f"{accuracy:2f}", step=step)
            print(f"{step} loss: {loss:2f} accuracy: {accuracy:2f} [{current} / {len(dataloader)-1}]")
    
    if batch > current:
      print(f"loss: {loss.item():2f} accuracy: {accuracy:2f} [{batch} / {len(dataloader)-1}]")

In [161]:
# Get cpu or gpu for training.
device = "cuda" if torch.cuda.is_available() else "cpu"

epochs = 3
loss_fn = nn.CrossEntropyLoss()
loss_fn = getattr(nn, 'CrossEntropyLoss')()
metric_fn = Accuracy(task="multiclass", num_classes=10).to(device)
model = MNISTClassifier().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [162]:
for e in range(epochs): 
  train(train_dataloader, model, loss_fn, metric_fn, optimizer, e, device)

0 loss: 2.355822 accuracy: 0.062500 [0 / 78]
1 loss: 2.233662 accuracy: 0.234375 [10 / 78]
2 loss: 2.198972 accuracy: 0.437500 [20 / 78]
3 loss: 2.148225 accuracy: 0.531250 [30 / 78]
4 loss: 2.142443 accuracy: 0.437500 [40 / 78]
5 loss: 2.043288 accuracy: 0.656250 [50 / 78]
6 loss: 1.961541 accuracy: 0.625000 [60 / 78]
7 loss: 1.848072 accuracy: 0.656250 [70 / 78]
loss: 1.745356 accuracy: 0.875000 [78 / 78]
0 loss: 1.795031 accuracy: 0.656250 [0 / 78]
2 loss: 1.619497 accuracy: 0.750000 [10 / 78]
4 loss: 1.558860 accuracy: 0.828125 [20 / 78]
6 loss: 1.502070 accuracy: 0.750000 [30 / 78]
8 loss: 1.479259 accuracy: 0.625000 [40 / 78]


KeyboardInterrupt: 

In [155]:
train_dataset = MNISTDatasetCsv('../data/train_data.csv.npy')
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)

5000

In [103]:
training_data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

In [104]:
for x in training_data:
  print(x[0].shape)
  break

torch.Size([1, 28, 28])


In [111]:
for s in train_dataset:
  print(s[0].shape)
  break

torch.Size([1, 28, 28])


In [54]:
train_tensor = torch.tensor(train_data).float().reshape(train_data.shape[0], 28, 28)

In [55]:
train_data.shape

(5000, 28, 28)

In [70]:
train_df = pd.read_csv('../data/train_data.csv', header=None, dtype=float)
train_df.shape[0]

5000